# ENEMDU 2021-2025 — Pipeline Medallón (Bronze → Silver → Gold)

**Proyecto:** UCUENCA-SABE · Empleabilidad de graduados universitarios

**Objetivos:**
1. Analizar la **empleabilidad de graduados universitarios** (2021-2025).
2. Identificar **sobrecalificación**: graduados ocupados en ocupaciones que no requieren título superior (CIUO-08, grandes grupos 4-9).
3. Generar **agregados listos para el dashboard** (KPIs, provincia, sexo/edad, rama de actividad, sobrecalificación por ocupación).

**Arquitectura:** este notebook es la puerta de entrada de todo el pipeline. Sólo orquesta —
la lógica pesada vive en `src/processing/` (`enemdu_schema_analyzer.py`, `enemdu_silver_builder.py`,
`enemdu_gold_builder.py`) para que sea reutilizable desde `scripts/run_enemdu_pipeline.py` y testeable
fuera del notebook. Ver `docs/README_ENEMDU_PIPELINE.md` para el detalle de cada decisión técnica.

**Descarga inteligente:** si Bronze ya existe localmente y trae los 5 años esperados, **no vuelve a
descargar de Kaggle**. Cambia `FORCE_DOWNLOAD = True` (celda 3) sólo si necesitas refrescar los datos.

In [1]:
import sys
from pathlib import Path

# Inyectar la raíz del proyecto al sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CONFIG
from src.ingestion.kaggle_downloader import download_and_organize_enemdu
from src.processing import (
    ENEMDUPaths,
    ENEMDUSchemaAnalyzer,
    ENEMDUSilverBuilder,
    ENEMDUGoldBuilder,
    load_silver,
)

print(f"✅ Proyecto cargado correctamente: {CONFIG['project']['name']}")
print(f"📁 Ruta raíz: {PROJECT_ROOT}")
print(f"📊 Sources disponibles: {list(CONFIG['sources'].keys())}")

✅ Proyecto cargado correctamente: ucuenca-sabe
📁 Ruta raíz: c:\Users\michu\mis-proyectos\ucuenca-sabe
📊 Sources disponibles: ['externas', 'internas', 'investigacion', 'enemdu']


## 1. Bronze — descarga condicionada desde Kaggle

Usa `ENEMDUPaths` (la misma utilidad de rutas que usan el analyzer, Silver y Gold) para saber
dónde vive Bronze, en vez de reconstruir la ruta a mano — así evitamos que el notebook y el
pipeline queden desincronizados si algún día cambia la convención de carpetas.

In [2]:
import re
import json
from datetime import datetime

YEARS_EXPECTED = list(range(2021, 2026))
FORCE_DOWNLOAD = False  # ⚙️ cambia a True para forzar una nueva descarga

paths = ENEMDUPaths.build(PROJECT_ROOT).ensure()
manifest_path = paths.bronze / ".manifest.json"


def bronze_status(paths: ENEMDUPaths, years_expected: list[int]) -> dict:
    """Verifica qué años de Bronze ya existen localmente en microdatos_csv/."""
    años_locales = []
    if paths.microdatos.exists():
        años_locales = sorted(
            int(m.group()) for f in paths.microdatos.glob("*.csv")
            if (m := re.search(r"(?:19|20)\d{2}", f.stem))
        )
    años_faltantes = [a for a in years_expected if a not in años_locales]

    última_descarga = None
    if manifest_path.exists():
        try:
            última_descarga = json.loads(manifest_path.read_text(encoding="utf-8")).get("última_descarga")
        except Exception:
            pass

    tamaño_mb = 0.0
    if paths.microdatos.exists():
        tamaño_mb = sum(f.stat().st_size for f in paths.microdatos.rglob("*.csv")) / 1e6

    return {
        "completo": len(años_faltantes) == 0 and len(años_locales) > 0,
        "años_locales": años_locales,
        "años_faltantes": años_faltantes,
        "tamaño_mb": tamaño_mb,
        "última_descarga": última_descarga,
    }


estado = bronze_status(paths, YEARS_EXPECTED)
print("=" * 70)
print("📦 ESTADO DE CACHÉ LOCAL (Bronze)")
print("=" * 70)
print(f"  Ubicación:       {paths.bronze}")
print(f"  Años locales:    {estado['años_locales'] or 'ninguno'}")
if estado["años_faltantes"]:
    print(f"  Años faltantes:  {estado['años_faltantes']}")
print(f"  Tamaño en disco: {estado['tamaño_mb']:.1f} MB")
if estado["última_descarga"]:
    print(f"  Última descarga: {estado['última_descarga']}")
print("=" * 70)

📦 ESTADO DE CACHÉ LOCAL (Bronze)
  Ubicación:       c:\Users\michu\mis-proyectos\ucuenca-sabe\data\bronze\externas\enemdu
  Años locales:    [2021, 2022, 2023, 2024, 2025]
  Tamaño en disco: 716.5 MB


In [3]:
if estado["completo"] and not FORCE_DOWNLOAD:
    print(f"✅ Bronze ya completo en caché ({estado['tamaño_mb']:.1f} MB) — se omite la descarga de Kaggle.")
    inventario = None
else:
    motivo = "forzado por FORCE_DOWNLOAD=True" if FORCE_DOWNLOAD else f"faltan años {estado['años_faltantes']}"
    print(f"🚀 Descargando desde Kaggle ({motivo})... esto puede tomar varios minutos.")
    try:
        inventario = download_and_organize_enemdu()
        manifest_path.write_text(
            json.dumps(
                {
                    "última_descarga": datetime.now().isoformat(),
                    "años": YEARS_EXPECTED,
                    "fuente": CONFIG["sources"]["enemdu"]["kaggle_dataset"],
                },
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        print("✅ Descarga completada y manifest actualizado.")
    except Exception as e:
        print(f"❌ Error en la descarga: {e}")
        raise

# Re-verificar tras la descarga (o confirmar que no hizo falta)
estado = bronze_status(paths, YEARS_EXPECTED)
print(f"\n📁 Bronze listo — años disponibles: {estado['años_locales']}")

✅ Bronze ya completo en caché (716.5 MB) — se omite la descarga de Kaggle.

📁 Bronze listo — años disponibles: [2021, 2022, 2023, 2024, 2025]


## 2. Objetivo 1 — Auditoría de esquemas (Bronze)

Detecta, por año, qué variables clave existen realmente en el CSV (vs. sólo documentadas en el
diccionario), y con qué nombre. Esto es lo que permite construir un Silver con esquema estable
aunque el INEC renombre columnas entre años.

In [4]:
analyzer = ENEMDUSchemaAnalyzer(paths=paths, years=estado["años_locales"] or None)
analyzer.analyze_all()

print(analyzer.summary().to_string(index=False))
print()
print(analyzer.audit().to_string(index=False))

report_path = analyzer.save_report()
print(f"\n📝 Reporte de auditoría guardado en: {report_path}")

10:53:30 | INFO    | src.processing.enemdu_schema_analyzer | Bronze ENEMDU: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\bronze\externas\enemdu
10:53:31 | ERROR   | src.processing.enemdu_schema_analyzer | Año 2021: el diccionario 'Diccionario de Datos_persona_anual_2021.xlsx' se leyó (153 variables documentadas) pero NINGUNA trae etiquetas de valor (código→significado) — probablemente sólo documenta nombre + descripción de campo, no categorías. TODO el mapeo semántico de sexo/condición de actividad/nivel de instrucción/etc. para este año depende ciegamente de los FALLBACK_* de enemdu_mappings.py, sin poder validarse contra un codebook real. Busca un 'Manual de usuario' o la versión .sav/.dta del microdato (trae las etiquetas embebidas) antes de confiar en las tasas derivadas de condicion_actividad.
10:53:31 | INFO    | src.processing.enemdu_schema_analyzer | Año 2021 → 151 cols CSV | 153 cols diccionario | cobertura 100.0%
10:53:31 | ERROR   | src.processing.enemdu_schema_analyzer | 

 anio                               csv                                   diccionario  filas  cols_csv  cols_dicc  cobertura_%  solo_en_csv  solo_en_dicc  cols_exclusivas  variables_con_etiquetas_valor  mapeo_semantico_validable
 2021 BDDenemdu_personas_2021_anual.csv  Diccionario de Datos_persona_anual_2021.xlsx 361790       151        153        100.0            0             2               12                              0                      False
 2022 BDDenemdu_personas_2022_anual.csv Diccionario de Datos_personas_anual_2022.xlsx 358096       139        141        100.0            0             2                0                              0                      False
 2023 BDDenemdu_personas_2023_anual.csv  Diccionario de Datos_persona_anual_2023.xlsx 345174       141        143        100.0            0             2                2                              0                      False
 2024 BDDenemdu_personas_2024_anual.csv  Diccionario de Datos_persona_anual_2024.xls

10:53:39 | INFO    | src.processing.enemdu_schema_analyzer | Reporte de esquemas guardado en c:\Users\michu\mis-proyectos\ucuenca-sabe\reports\schemas\schema_analysis_20260907.json


 anio  filas_totales  filas_muestra  columnas  peso_mb encoding sep  pct_nulos_medio  cols_100pct_nulas                                                         llave_detectada  dups_llave  edad_min  edad_max  fexp_presente criticas_faltantes
 2021         361790          50000       151   155.36    utf-8   ;              0.0                  0 id_persona + id_hogar + conglomerado + vivienda + hogar + p01 + periodo           0       0.0      98.0           True                  —
 2022         358096          50000       139   145.27    utf-8   ;              0.0                  0 id_persona + id_hogar + conglomerado + vivienda + hogar + p01 + periodo           0       0.0      98.0           True                  —
 2023         345174          50000       141   141.42    utf-8   ;              0.0                  0 id_persona + id_hogar + conglomerado + vivienda + hogar + p01 + periodo           0       0.0      98.0           True                  —
 2024         341394          50

## 3. Objetivo 2 (base) — Construcción de Silver

Unifica los 5 años en un único esquema (alias resueltos por año, códigos → etiquetas desde el
diccionario, no-respuesta → `NaN`, columnas ausentes → `NaN` nunca `0`). No filtra graduados ni
aplica PET todavía: Silver debe servir a *cualquier* módulo del dashboard, no sólo a empleabilidad.

In [5]:
silver_builder = ENEMDUSilverBuilder(analyzer=analyzer)
silver_result = silver_builder.build()

print(silver_result.quality.to_string(index=False))

silver_builder.save(silver_result)
print(f"\n💾 Silver guardado en: {paths.silver / 'enemdu_unificado.parquet'}")

10:53:39 | INFO    | src.processing.enemdu_silver_builder | Año 2021 → 27/30 variables canónicas resueltas
10:53:50 | INFO    | src.processing.enemdu_silver_builder | Año 2022 → 27/30 variables canónicas resueltas
10:54:00 | INFO    | src.processing.enemdu_silver_builder | Año 2023 → 29/30 variables canónicas resueltas
10:54:10 | INFO    | src.processing.enemdu_silver_builder | Año 2024 → 27/30 variables canónicas resueltas
10:54:19 | INFO    | src.processing.enemdu_silver_builder | Año 2025 → 27/30 variables canónicas resueltas


 anio  filas  poblacion_expandida  edad_media  %_nulos_nivel_instruccion  %_nulos_condicion_actividad  %_nulos_ciuo  %_nulos_ingreso  %_nulos_fexp  fexp_media  fexp_mediana  fexp_max  graduados_superior  ocupados  desempleados mapeo_educacion mapeo_educacion_variable mapeo_condact                                      dedup_llave  dedup_id_persona_utilizable  dedup_filas_eliminadas
 2021 361790           17816201.0        33.7                       6.36                          0.0         49.68            60.54           0.0        49.2          27.3    1546.0               69807    169619         11156        fallback                     p10a      fallback [mes, conglomerado, vivienda, hogar, id_persona]                         True                       0
 2022 358096           18060182.0        34.7                       6.04                          0.0         49.31            58.38           0.0        50.4          25.7    1411.1               70271    171800          9300      

10:54:48 | INFO    | src.processing.enemdu_silver_builder | Silver guardada: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\silver\enemdu\enemdu_unificado.parquet (1741240 filas, 52 cols, 36.8 MB)



💾 Silver guardado en: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\silver\enemdu\enemdu_unificado.parquet


### 3.1 Checkpoint — población expandida por año (ANTES de pasar a Gold)

**Por qué este checkpoint existe:** `_clean_types()` convertía `id_persona` (código INEC de 22
dígitos: provincia+cantón+parroquia+zona+sector+vivienda+hogar+persona) a numérico junto con el
resto de columnas. Un `float64` sólo tiene ~15-17 dígitos de precisión: un id de 22 dígitos pierde
precisión al redondearse y **personas distintas colapsaban al mismo valor** (2021: 361,790
`id_persona` únicos de verdad → sólo 44,900 tras la conversión). La deduplicación de
`_deduplicate_per_year` usa `id_persona` como parte de su llave, así que borraba como "duplicados"
personas reales: **71% de las filas de 2021** (258,302 de 361,790), y proporciones similares el
resto de años. Con eso, la población expandida daba ~4.7-5.2M por año en vez de los ~17.8-18.9M
reales (Ecuador tiene ~15-19M de habitantes en este período).

**La corrección** (ya aplicada en `enemdu_silver_builder.py`): `id_persona` se mantiene como texto
exacto — es un identificador, nunca se usa en aritmética, así que no hay motivo para convertirlo a
número. Este checkpoint queda para que si algo similar se reintroduce en el futuro (por ejemplo, un
año nuevo con un formato de `id_persona` distinto), se detecte de inmediato: si la población
expandida de algún año cae fuera de 15-19M, o si `dedup_filas_eliminadas` deja de ser 0, algo se
rompió en la resolución de columnas o en la deduplicación.

In [6]:
POBLACION_ECUADOR_MIN = 15_000_000
POBLACION_ECUADOR_MAX = 19_000_000

chk = silver_result.quality[[
    "anio", "filas", "poblacion_expandida",
    "dedup_id_persona_utilizable", "dedup_filas_eliminadas",
]].copy()
chk["pct_eliminado_dedup"] = (
    100 * chk["dedup_filas_eliminadas"] / (chk["filas"] + chk["dedup_filas_eliminadas"])
).round(2)
chk["poblacion_en_rango_ecuador"] = chk["poblacion_expandida"].between(
    POBLACION_ECUADOR_MIN, POBLACION_ECUADOR_MAX)

print("=" * 100)
print(f"✅ CHECKPOINT — Población expandida por año (ponderada por fexp), ANTES de construir Gold")
print("=" * 100)
print(chk.to_string(index=False))

fuera_de_rango = chk[~chk["poblacion_en_rango_ecuador"]]
if not fuera_de_rango.empty:
    print(f"\n⚠️  {len(fuera_de_rango)} año(s) con población expandida FUERA de "
          f"{POBLACION_ECUADOR_MIN/1e6:.0f}-{POBLACION_ECUADOR_MAX/1e6:.0f}M "
          f"(rango plausible para Ecuador en este período):")
    print(fuera_de_rango[["anio", "poblacion_expandida"]].to_string(index=False))
    print("   → No sigas a Gold sin revisar: puede ser un problema de fexp o de deduplicación "
          "(ver celda de arriba sobre el bug de id_persona).")
else:
    print(f"\n✅ Los {len(chk)} años caen dentro de "
          f"{POBLACION_ECUADOR_MIN/1e6:.0f}-{POBLACION_ECUADOR_MAX/1e6:.0f}M — "
          "población expandida consistente con Ecuador.")

if (chk["dedup_filas_eliminadas"] > 0).any():
    print("\n⚠️  Hay años con filas eliminadas por deduplicación — antes de continuar confirma "
          "que 'id_persona' se resolvió como texto exacto (dedup_id_persona_utilizable=True) "
          "y no colapsó personas distintas.")
else:
    print("✅ 0 filas eliminadas por deduplicación en todos los años (esperado: no hay "
          "duplicados reales en Bronze; ver celda de arriba).")

assert fuera_de_rango.empty, (
    "Población expandida fuera del rango esperado para Ecuador — no continúes a Gold sin revisar."
)

✅ CHECKPOINT — Población expandida por año (ponderada por fexp), ANTES de construir Gold
 anio  filas  poblacion_expandida  dedup_id_persona_utilizable  dedup_filas_eliminadas  pct_eliminado_dedup  poblacion_en_rango_ecuador
 2021 361790           17816201.0                         True                       0                  0.0                        True
 2022 358096           18060182.0                         True                       0                  0.0                        True
 2023 345174           18307511.0                         True                       0                  0.0                        True
 2024 341394           18600313.0                         True                       0                  0.0                        True
 2025 334786           18855042.0                         True                       0                  0.0                        True

✅ Los 5 años caen dentro de 15-19M — población expandida consistente con Ecuador.
✅ 0 filas el

## 4. Objetivos 1-3 — Construcción de Gold (empleabilidad, sobrecalificación, agregados)

Aquí se aplican los filtros analíticos (PET ≥15, graduados) y la ponderación por `fexp`. La capa
Gold ya sale en forma de tablas de hechos (numerador/denominador, no sólo tasas) listas para
Power BI o para `visualizacion/scripts/generate_dashboard_data.py`.

- `empleabilidad_graduados` → tabla de hechos, granularidad persona-año
- `kpi_anual` → serie 2021-2025 (desempleo, empleo adecuado, subempleo, **sobrecalificación**)
- `empleabilidad_provincia`, `empleabilidad_sexo_edad`, `graduados_rama_actividad` → agregados dimensionales
- `sobrecalificacion_ocupacion` → tasa de sobrecalificación por gran grupo CIUO-08

In [7]:
silver = load_silver(paths)
gold_builder = ENEMDUGoldBuilder(silver, paths=paths)
gold_result = gold_builder.build()

print("📈 KPI anual (empleabilidad + sobrecalificación):")
print(gold_result.kpi_anual.to_string(index=False))

print("\n🎓 Sobrecalificación por ocupación (CIUO-08, grandes grupos 4-9):")
print(gold_result.sobrecalificacion_ocupacion.to_string(index=False))

gold_builder.save(gold_result, export_csv=True)
print(f"\n💾 Gold guardado en: {paths.gold}")
print(f"💾 CSV para Power BI en: {paths.reports / 'powerbi'}")

10:54:53 | INFO    | src.processing.enemdu_gold_builder | PET (>= 15 años): 1362078 de 1741240 filas
10:54:53 | INFO    | src.processing.enemdu_gold_builder | Construyendo Gold sobre 1362078 filas de Silver...


📈 KPI anual (empleabilidad + sobrecalificación):
          segmento  anio  n_muestral  poblacion       pea  ocupados  desempleados  empleo_adecuado  subempleados  sobrecalificados  ocupados_con_ciuo  tasa_participacion  tasa_empleo  tasa_desempleo  tasa_empleo_adecuado  tasa_subempleo  tasa_sobrecalificacion  ingreso_laboral_medio  confiable
           general  2021      277290 12671329.0 8362453.0 7924595.0      437857.0        2718798.0     1942602.0          817294.0          7924595.0               66.00        94.76            5.24                 32.51           23.23                   10.31                 437.23       True
           general  2022      278069 12850763.0 8471135.0 8102557.0      368579.0        2914727.0     1882915.0          799569.0          8102557.0               65.92        95.65            4.35                 34.41           22.23                    9.87                 457.70       True
           general  2023      269914 13032742.0 8434079.0 8109647.

10:55:01 | INFO    | src.processing.enemdu_gold_builder | Gold guardada: 7 tablas en c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu



💾 Gold guardado en: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu
💾 CSV para Power BI en: c:\Users\michu\mis-proyectos\ucuenca-sabe\reports\powerbi


## 5. Siguiente paso — alimentar el dashboard

Este notebook deja todo lo necesario en `data/gold/enemdu/*.parquet`. El dashboard HTML **no**
debe leer ni agregar datos por su cuenta: `visualizacion/scripts/generate_dashboard_data.py` lee
Gold y escribe `visualizacion/static/data.json`, que es lo único que carga el HTML.

```bash
python visualizacion/scripts/generate_dashboard_data.py
```

Ejecuta esa celda (o el script desde terminal) cada vez que vuelvas a correr este notebook con
datos nuevos.

In [8]:
# Descomenta para regenerar el JSON del dashboard automáticamente:
# import subprocess
# subprocess.run([sys.executable, str(PROJECT_ROOT / "visualizacion" / "scripts" / "generate_dashboard_data.py")], check=True)